# HeritAgent — EXP-0001 GPU'suz prova (Colab, CPU çalışma zamanı)

A100'e para harcamadan **önce** bu notebook'u bir **CPU** çalışma zamanında çalıştırın. Pilot ve koşu notebook'ları, aynı kodun (`CODE_REF`) ve modelin (`MODEL`) provası geçmediyse ilk dakikalarda durur.

**Ne yapar:** pilotun tamamını GPU'suz ve modelsiz oynatır. Pilotla aynı adımlar şunlar:
- aynı kurulum: depodaki kilitli ortam (sabit Python ve hash'li paketler; deneyin komutları Colab'ın kendi Python'unu kullanmaz);
- aynı ALFWorld verisi ve hash denetimi;
- 30 pilot oyunu, 16 worker sürecinde;
- aynı kayıt defteri (ledger), bölüm kayıtları ve Drive yansısı;
- modelin kendi tokenizer'ı (yalnızca tokenizer dosyaları indirilir, ağırlıklar indirilmez);
- aynı rapor.

Model sunucusunun yerinde bu makinede çalışan taklit bir sunucu vardır: vLLM'in cevapları gibi cevaplar ve bazı bölümleri ALFWorld'ün kendi uzman planıyla kazanır. Yolda bir kesinti çıkarılır: pilot yarıda durur, Drive'daki kopyadan geri yüklenir ve kaldığı yerden biter. Sonunda rapor, taklit sunucunun yaptığıyla karşılaştırılır.

**Hiçbir şey ölçmez.** Prova raporu bir pilot raporu değildir ve hiçbir modeli ölçmez. Yalnızca kodun bu makinede baştan sona çalıştığını gösterir.

**Tahmini süre ve maliyet (M):** kurulum ~10 dk, veri ~5 dk, prova ~10–25 dk. CPU çalışma zamanı saatte A100'ün çok küçük bir kesrini harcar.

**Nasıl çalıştırılır:**
1. Çalışma zamanı > Çalışma zamanı türünü değiştir > **CPU**. Varsa "Yüksek RAM"i seçin.
2. Çalışma zamanı > **Tümünü çalıştır**. Drive izni isteyince onaylayın.
3. Sonunda "PROVA GEÇTİ" yazar ve kayıt Drive'da `heritagent/rehearsal/` altına yazılır. Ardından pilot notebook'unu A100'de, `CODE_REF`'e son satırdaki commit'i yazarak çalıştırın.
4. Bir hücre hata verirse hata metnini bana gönderin; A100'e hiç para harcanmamış olur.

In [ ]:
# Ayarlar (pilot ve koşu notebook'larıyla aynı CODE_REF ve MODEL)
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
WORKERS = 16               # pilotla aynı sayıda worker süreci
MODEL = "Qwen/Qwen3.8-27B"
AUTO_RELEASE = True        # bitince ya da bir hata olursa çalışma zamanını bırak

In [ ]:
# 0) Bir hücre hata verirse "Tümünü çalıştır" orada durur. AUTO_RELEASE açıksa sunucu günlüğü
# Drive'a kopyalanır, Drive boşaltılır ve çalışma zamanı bırakılır: hatadan sonra boşta
# bekleyen çalışma zamanı da CU harcar. Bir deney komutu hâlâ çalışıyorsa ya da son kopyası
# Drive'a alınamamış bir koşu varsa çalışma zamanı BIRAKILMAZ (o iş yalnızca bu makinededir).
import glob, os, shutil, time
from google.colab import drive, runtime as colab_runtime  # kurulumdan önce yüklenir


def unmirrored():
    """Son kopyası Drive'a alınamamış yerel koşu klasörleri (ADR-0008 D5)."""
    return sorted(os.path.dirname(p) for p in glob.glob("/content/*/UNMIRRORED.json"))


def running_commands():
    """Bu makinede hâlâ çalışan run/pilot/snapshot/rehearse komutlarının süreç numaraları. Çekirdek
    yeniden başladıysa önceki komut kendiliğinden durur, ama süren bölümlerini kaydedip son
    kopyayı alana dek (birkaç dakika) çalışır."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit() or int(entry) == os.getpid():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"evaluation.exp0001.cli" in argv:
            command = argv[argv.index(b"evaluation.exp0001.cli") + 1:][:1]
            if command and command[0] in (b"run", b"pilot", b"snapshot", b"rehearse"):
                found.append(int(entry))
    return sorted(found)


def release():
    """Drive'ı boşaltır ve çalışma zamanını bırakır; bir komut hâlâ çalışıyorsa ya da aynalanmamış bir
    koşu varsa bırakmaz."""
    busy = running_commands()
    if busy:
        print("ÇALIŞMA ZAMANI BIRAKILMADI: bir deney komutu hâlâ çalışıyor (süreç", busy, "). Çekirdek "
              "yeniden başladıysa bu önceki komuttur: süren bölümlerini kaydedip son kopyayı "
              "alıyor.")
        print("Birkaç dakika bekleyin, sonra o hücreyi yeniden çalıştırın (kaldığı yerden "
              "sürer) ya da release() çalıştırın.")
        print("15 dakikada bitmezse: import signal; os.killpg(<süreç>, signal.SIGTERM) ile "
              "sonlandırın. Son kopyadan sonraki kayıtlar bu makinede kalır (UNMIRRORED.json); "
              "ardından cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>) "
              "çalıştırın (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + "
              "'/mirror'). Runtime > Disconnect and delete runtime ise o kayıtları siler.")
        return
    kept = unmirrored()
    if kept:
        print("ÇALIŞMA ZAMANI BIRAKILMADI: son kopyası Drive'a alınamamış iş var:", kept)
        print("1) Drive'ı yeniden bağlayın: drive.mount('/content/drive', force_remount=True)")
        print("2) Her klasör için: cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>)")
        print("   (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + '/mirror')")
        print("3) Kalan hücreleri (durum, rapor kopyası) çalıştırın; sonra release() çalıştırın: "
              "Drive'ı boşaltır ve çalışma zamanını bırakır.")
        return
    try:  # Drive'a yazılanlar makine kapanmadan yüklensin; en fazla 10 dakika beklenir
        drive.flush_and_unmount(timeout_ms=600_000)
    except BaseException as exc:  # bağlanmamış, takılmış ya da durdurulmuş: çalışma zamanı yine bırakılır
        print("Drive boşaltılamadı:", repr(exc))
    print("Çalışma zamanı bırakılıyor.")
    try:
        colab_runtime.unassign()
    except Exception as exc:
        print("Çalışma zamanı bırakılamadı; bağlantıyı kendiniz kesin:", exc)


def _on_error(shell, etype, value, tb, tb_offset=None):
    try:
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
        if not AUTO_RELEASE:
            print("AUTO_RELEASE = False: çalışma zamanı bağlı kalıyor; işiniz bitince bağlantıyı kesin.")
            return
        try:
            if os.path.isdir(DRIVE_ROOT) and os.path.exists("/content/vllm-server.log"):
                saved = f"{DRIVE_ROOT}/logs/vllm-server-{time.strftime('%Y%m%d-%H%M%S')}.log"
                os.makedirs(os.path.dirname(saved), exist_ok=True)
                shutil.copy("/content/vllm-server.log", saved)
                print("Sunucu günlüğü Drive'da:", saved)
        except Exception as exc:
            print("Sunucu günlüğü Drive'a kopyalanamadı:", exc)
        if etype.__name__ != "Bitti":
            print("Bir adım başarısız oldu. Hata metnini bana gönderin.")
    finally:  # günlükteki ya da Drive'daki bir hata çalışma zamanının bırakılmasını engellemez
        if AUTO_RELEASE:
            release()


get_ipython().set_custom_exc((Exception,), _on_error)

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)


class Bitti(Exception):
    pass

In [ ]:
# 3) Kod: depoyu klonla ve CODE_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)


def checkout(ref):
    target = f"origin/{ref}" if subprocess.run(
        ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
        capture_output=True).returncode == 0 else ref
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
    return subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                          text=True, check=True).stdout.strip()


COMMIT = checkout(CODE_REF)
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 4) Kurulum (~5-10 dk): deneyin komutları Colab'ın kendi Python'unu ve paketlerini kullanmaz.
# Her oturumda depodaki environments/exp0001-colab/ dosyalarına göre (CODE_REF'teki hâlleriyle)
# AYNI ortam kurulur: hash'iyle uv (uv-requirements.txt), sabit Python (python-version), bütün
# paketlerin hash'li kilidi (requirements.lock: vLLM 0.29.0, torch 2.13.0 CUDA 12.9 yapısı,
# transformers, tokenizers, huggingface-hub, xgrammar, llguidance, jinja2, ALFWorld 0.4.2,
# TextWorld 1.7.0 ve bütün bağımlılıkları) ve kaynaktan derlenen iki paketin hash'li derleme
# araçları (build-constraints.txt). Prova ile koşu böylece aynı ortamda çalışır; sonraki hücre
# ortamı denetler. Koşu başladığı sürümleri kaydeder; başka sürümlerle devam etmez (ADR-0008 D4).
import hashlib, os, signal, subprocess, sys

ENV = f"{REPO}/environments/exp0001-colab"
VENV = "/content/heritagent-venv"
PY = f"{VENV}/bin/python"  # deneyin komutları bu ortamın Python'uyla çalışır


def command_environment():
    """Komutların ortam değişkenleri: çekirdeğin Python ve matplotlib ayarları (PYTHONPATH,
    MPLBACKEND gibi) bu ortama geçmez; ortamın bin klasörü PATH'te önce gelir (vllm serve)."""
    kept = {k: v for k, v in os.environ.items() if not k.startswith(("PYTHON", "MPL"))}
    return {**kept, "VIRTUAL_ENV": VENV, "PATH": f"{VENV}/bin:{os.environ['PATH']}",
            "MPLBACKEND": "Agg", "PYTHONNOUSERSITE": "1"}


def vllm_servers():
    """Bu makinede çalışan `vllm serve` süreçlerinin numaraları."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"serve" in argv[1:3] and any(os.path.basename(arg) == b"vllm" for arg in argv[:2]):
            found.append(int(entry))
    return sorted(found)


def install():
    """Ortamı kurar. Bir deney komutu çalışırken hiçbir şeye dokunmaz; eski ortamdan bir vLLM
    sunucusu çalışırken ortamı değiştirmez (kurulan ortam, kurulduğu dosyaların damgasını taşır)."""
    busy = running_commands()
    if busy:  # yeniden başlayan bir çekirdek, hâlâ çalışan bir komutun ortamına dokunmaz
        raise RuntimeError(f"Bir deney komutu hâlâ çalışıyor (süreç {busy}); o bitince bu "
                           f"hücreyi yeniden çalıştırın.")
    with open(f"{ENV}/python-version", encoding="utf-8") as handle:
        python = handle.read().strip()
    digest = hashlib.sha256()
    for name in ("python-version", "uv-requirements.txt", "requirements.lock",
                 "build-constraints.txt"):
        with open(f"{ENV}/{name}", "rb") as handle:
            digest.update(handle.read())
    stamp = f"{VENV}/heritagent-environment.sha256"
    try:
        with open(stamp, encoding="utf-8") as handle:
            same = handle.read().strip() == digest.hexdigest()
    except OSError:
        same = False
    servers = vllm_servers()
    if servers and not same:
        raise RuntimeError(f"Ortam değişecek ama bir vLLM sunucusu hâlâ eski ortamdan çalışıyor "
                           f"(süreç {servers}): os.kill(<süreç>, signal.SIGTERM) ile durdurup bu "
                           f"hücreyi yeniden çalıştırın.")
    env = command_environment()  # kaynaktan derlenen paketler de çekirdeğin ayarlarını görmez
    uv = [sys.executable, "-m", "uv"]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--require-hashes",
                    "--only-binary", ":all:", "-r", f"{ENV}/uv-requirements.txt"], check=True)
    try:
        with open(f"{VENV}/pyvenv.cfg", encoding="utf-8") as handle:
            made = dict(line.split(" = ", 1) for line in handle.read().splitlines()
                        if " = " in line)
    except OSError:
        made = {}
    if made.get("version_info") != python or not os.path.exists(PY):
        subprocess.run(uv + ["venv", "--clear", "--managed-python", "--python", python, VENV],
                       check=True, env=env)
    build = ["--build-constraints", f"{ENV}/build-constraints.txt"]
    subprocess.run(uv + ["pip", "sync", "--python", PY, "--require-hashes", "--index-strategy",
                         "unsafe-best-match", *build, f"{ENV}/requirements.lock"],
                   check=True, env=env)
    subprocess.run(uv + ["pip", "install", "--python", PY, "--no-deps", *build, "-e", REPO],
                   check=True, env=env)
    subprocess.run(uv + ["pip", "check", "--python", PY], check=True, env=env)  # bağımlılıklar
    with open(stamp, "w", encoding="utf-8") as handle:
        handle.write(digest.hexdigest() + "\n")


install()


def cli(*args):
    """Deneyin bir komutunu çalıştırır ve çıktısını canlı yazar. Komut başarısız olursa hata
    verir; böylece "Tümünü çalıştır" sonraki (para harcayan) hücrelere geçmez.
    Durdur düğmesi: komut, süre sınırındaki gibi durur (sürmekte olan bölümler kaydedilir,
    son kopya Drive'a alınır); ikinci basış onu hemen sonlandırır."""
    command = [PY, "-u", "-m", "evaluation.exp0001.cli", *map(str, args)]
    # kendi süreç grubunda: durdurma sinyali ona yalnızca buradan, bir kez ulaşır; çekirdek
    # yeniden başlarsa ya da ölürse komut da aynı şekilde durur (HERITAGENT_STOP_WITH_PARENT)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True,
                          env={**command_environment(),
                               "HERITAGENT_STOP_WITH_PARENT": str(os.getpid())}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            print("Durduruluyor: süren bölümler kaydediliyor ve son kopya alınıyor (birkaç "
                  "dakika sürebilir). Hemen sonlandırmak için yeniden basın.", flush=True)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
                print("Sonlandırıldı. Bu bir run, pilot ya da snapshot komutuysa son kopyadan "
                      "sonraki kayıtlar yalnızca bu makinede olabilir ve UNMIRRORED.json "
                      "bırakılmıştır (unmirrored() ile bakın): çalışma zamanı, cli('snapshot', "
                      "'--run-dir', <klasör>, '--mirror-dir', <ayna>) çalışana dek "
                      "bırakılmaz.", flush=True)
            raise
    if process.returncode == 3:  # cli.EXIT_UNMIRRORED
        raise RuntimeError(f"`cli {args[0]}`: son kopya Drive'a alınamadı (UNMIRRORED); en yeni "
                           f"kayıtlar yalnızca bu makinede. Çalışma zamanı bırakılmayacak.")
    if process.returncode != 0:
        raise RuntimeError(f"`cli {args[0]}` başarısız oldu (çıkış kodu {process.returncode}); "
                           f"sonraki hücreler çalıştırılmadı.")

In [ ]:
# 4c) Ortam kilitli olanın aynısı mı (Python, her paket bir kez, vllm bu ortamdan; değilse
# veri ve model indirilmeden durulur)? Kurulan sürümler açılıyor mu? Bu makinede GPU yok: CUDA
# beklenmez, vLLM'in sunucuyu açması A100'de denetlenir. torch, transformers, tokenizers ya da
# matplotlib açılmazsa prova burada durur.
cli("env-check", "--repo", REPO)
check = subprocess.run([PY, "-c", "import torch, transformers, tokenizers, matplotlib; "
                        "print('torch', torch.__version__, 'transformers', transformers.__version__)"],
                       capture_output=True, text=True, env=command_environment())
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0, "Kurulum denetimi başarısız: torch/transformers/matplotlib açılmadı."
vllm_check = subprocess.run([PY, "-c", "import vllm; print('vllm', vllm.__version__)"],
                            capture_output=True, text=True, env=command_environment())
print(vllm_check.stdout or "vLLM bu CPU makinesinde açılmadı (A100'de denetlenir):",
      vllm_check.stderr[-1500:] if vllm_check.returncode else "")

In [ ]:
# 5) Drive denetimi (saniyeler): yazma, üzerine yazma, geri okuma, küçük dosya süresi.
import time
STAMP = time.strftime("%Y%m%d-%H%M%S")
REHEARSAL_DRIVE = f"{DRIVE_ROOT}/rehearsal/{STAMP}"
cli("mirror-check", "--mirror-dir", f"{REHEARSAL_DRIVE}/mirror-check")

In [ ]:
# 6) ALFWorld verisi (~2-5 dk): manifestlerin kurulduğu sürüm 0.2.2, sha256 ile doğrulanır.
# (`alfworld-download` başka bir sürüm indirir; onunla hiçbir oyun açılmaz.) Ardından üç
# manifestin (bölmeler, pilot, fayda bankası) 1.490 oyununun her biri hash'iyle denetlenir;
# tutmazsa model indirilmeden durur.
import os
os.environ["ALFWORLD_DATA"] = "/content/alfworld-data"
DATA_ROOT = "/content/alfworld-data/json_2.1.1"
cli("fetch-data", "--target", "/content/alfworld-data")
cli("verify-data", "--data-root", DATA_ROOT, "--repo", REPO)

In [ ]:
# 7) Prova (~10-25 dk): pilotun tamamı taklit sunucuya karşı, bir kesinti ve bir geri yüklemeyle.
import json, os, shutil
revision_file = f"{DRIVE_ROOT}/model-revision-{MODEL.replace('/', '--')}.txt"
revision = open(revision_file).read().strip() if os.path.exists(revision_file) else None
cli("rehearse", "--model", MODEL, *(["--revision", revision] if revision else []),
    "--data-root", DATA_ROOT, "--repo", REPO, "--workers", WORKERS,
    "--out-dir", f"/content/rehearsal-{STAMP}", "--mirror-dir", f"{REHEARSAL_DRIVE}/mirror")
passed = json.load(open(f"/content/rehearsal-{STAMP}/rehearsal-passed.json"))
if revision is None:  # pilot ve koşu bu revizyonu kullanır
    open(revision_file, "w").write(passed["model"]["revision"])
target = f"{DRIVE_ROOT}/rehearsal/passed-{COMMIT}-{MODEL.replace('/', '--')}.json"
shutil.copy(f"/content/rehearsal-{STAMP}/rehearsal-passed.json", target)
print("PROVA GEÇTİ:", COMMIT[:12], MODEL, "@", passed["model"]["revision"][:12], "| Python",
      passed["python"], "| kayıt:", target)
print("Şimdi pilot notebook'unu A100'de aynı MODEL ile çalıştırın. main bu arada ilerleyebilir; "
      "provayı yinelememek için orada CODE_REF'e bu commit'i yazın:")
print(f'CODE_REF = "{COMMIT}"')

In [ ]:
# Son) Drive'ı boşalt ve çalışma zamanını bırak (CPU çalışma zamanı da, az da olsa, CU harcar).
# İncelemek isterseniz AUTO_RELEASE = False yapın. (Bir hücre hata verirse 0. hücre aynısını yapar.)
if AUTO_RELEASE:
    release()